In [ ]:
!pip install import-ipynb

# Μέρος πρώτο: Δημιουργία Webcrawler 


In [21]:
import requests
import json
from bs4 import BeautifulSoup

### Συνάρτηση για συλλογή άρθρων 

Μέσω της BeautifulSoup η συνάρτηση WebCrawler συλλέγει το url, το κείμενο στην ετικέτα h1 για τον τίτλο και κάθε κείμενο στις ετικέτες p για το περιεγχόμενο. 

In [24]:
def WebCrawler(Url):
    try:
        OpenURL = requests.get(Url)
        Redirected = OpenURL.url
        soup = BeautifulSoup(OpenURL.text, 'html.parser')

        Title = soup.find('h1').string
        Content = "".join(p.text.strip() for p in soup.find_all('p'))
        
        print("Scraped Successfully")
        return {
            'Url': Redirected,
            'Title': Title,
            'Content': Content
        }
    except Exception as e:
        print(f"Failed to scrape {e}")


### Συνάρτηση για αποθήκευση των άρθρων σε αρχείο json

Αρχικά αποθηκεύει σε έναν πίνακα 10 τυχαία url, καλεί την συνάρτηση WebCrawler για κάθε url και αποθηκεύει τα δεδομένα σε ένα αρχείο JSON

In [27]:
def main():
    Urls = ["https://en.wikipedia.org/wiki/Special:Random" for _ in range(10)]
    
    Results = []
    for Url in Urls:
        
        Data = WebCrawler(Url)
        if Data:
            Results.append(Data)

    with open("Data.json", "w") as File:
        json.dump(Results, File, indent=10)
    print("Data successfully saved on Data.json")
    
if __name__ == "__main__":
    main()

Scraped Successfully
Scraped Successfully
Scraped Successfully
Scraped Successfully
Scraped Successfully
Scraped Successfully
Scraped Successfully
Scraped Successfully
Scraped Successfully
Scraped Successfully
Data successfully saved on Data.json


# Μέρος δεύτερο:  Προεπεξεργασία κειμένου

In [29]:
import json
import string
import nltk

Το περιγχόμενο των άρθρων χωρίζεται σε ξεχοριστές εκφράσεις.
Ο πίνακας FirstFilter αποθηκεύει κάθε έκφραση εξερώντας τα σημεία στίξης 
Ο πίνακας SecondFilter αποθηκεύει τις εκφράσεις του πίνακα FirstFilter εξερώντας τα stopwords.
Το περιεχόμενο του SecondFilter αποθηκεύεται στον πίνακα Results o οποίος αποθηκεύεται σε ένα αρχείο json.

In [33]:
def main():
    with open('Data.json', 'r') as File:
        Data = json.load(File)

    Results = []
    Stopwords = nltk.corpus.stopwords.words('english')
    Punctuation = string.punctuation
    for Json in Data:
        Words = Json['Content'].split()
        Words = [w.lower() for w in Words]
        FirstFilter = [''.join(letter for letter in Word if letter not in Punctuation) for Word in Words]
        SecondFilter = [Word for Word in FirstFilter if Word.lower() not in Stopwords]
        Results.append(SecondFilter)

    with open("Data2.json", "w") as File:
        json.dump(Results, File, indent=4)
    print("Data successfully saved on Data2.json")
    
if __name__ == "__main__":
    main()

Data successfully saved on Data2.json


# Μέρος τρίτο: Ευρετήριο

Η συνάρτηση InvertedIndex δημιουργεί ανεστραμμένο ευρετήριο χρησιμοποιώντας defaultdict(list).

H defaultdict δημιουργεί λεξικό όπου κάθε τιμή είναι λίστα. Η enumerate επιστρέφει αριθμητικό δείκτη (WikiId) και λίστα λέξεων (Words) από το αρχείο Data2.json. Για κάθε Word στη λίστα αν το WikiId δεν υπάρχει, το προσθέτει στη λίστα του ευρετηρίου. Τελος, επιστρέφει το λεξικό ως dict.

In [37]:
import json
from collections import defaultdict

def InvertedIndex(Data):
    
    II = defaultdict(list)
    for WikiId, Words in enumerate(Data):
        for Word in Words:
            if WikiId not in II[Word]:
                II[Word].append(WikiId)

    return dict(II)

Η συνάρτηση main φορτώνει δεδομένα από το αρχείο Data2.json, δημιουργεί το ανεστραμμένο ευρετήριο με τη συνάρτηση InvertedIndex και αποθηκεύει το αποτέλεσμα στο Data3.json. Τέλος, εμφανίζει μήνυμα επιτυχίας.

In [40]:
def main():
    with open('Data2.json', 'r') as File:
        Data = json.load(File)
    
    Results = []
    Results = InvertedIndex(Data)
    
    with open("Data3.json", "w") as File:
        json.dump(Results, File, indent=4)
    print("Inverted Indexes have succefully been saved on Data3.json")
        
if __name__ == "__main__":
    main()

Inverted Indexes have succefully been saved on Data3.json


# Μέρος τέταρτο: Μηχανή αναζήτησης

In [43]:
import json
import re

# Βιβλιοθήκη sklearn για τον αλγοριθμο TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Βιβλιοθήκη rank_bm25 για τον αλγοριθμο Probabilistic retrieval
from rank_bm25 import BM25Okapi


#### Υλοποίηση αλγορίθμου Boolean Retrieval

In [46]:
def BooleanRetrieval(Query, Records, SearchWords, Data):

    # Λίστα με τους αποδεκτούς λογικούς τελεστές
    Operators = ['and', 'not', 'or']

    # Αρχικοποίηση πίνακα αποτελεσμάτων και μεταβλητής για την αποθήκευση του τελεστή
    Results = []
    Op = ''

    # Επανάληψη για κάθε έκφραση του ερωτήματος
    for Word in Query:

        # Αποθήκευση τελεστή
        if Word in Operators:
            Op = Word

        # Κλήση συνάρτησης SearchKeyword για την λέξη κλειδί και αποθήκευση της τομής,ένωσης ή συμπληρώματος των αποτελεσμάτων 
        else:
            WordResults = SearchKeyword(Word)
            if Op == 'not':
                WordResults = list(set(range(Records)) - set(WordResults))
            if not Results:
                Results.append(WordResults)
            elif Op == 'and':
                Results[-1] = list(set(Results[-1]) & set(WordResults))
            elif Op == 'or':
                Results[-1] = list(set(Results[-1]) | set(WordResults))
            else:
                Results.append(WordResults)

    # Ένωαη αποτελεσμάτων σε έναν πίνακα
    Found = Results
    if Found:
        Found = list(set.union(*[set(f) for f in Found]))
    else:
        Found = [] 

    # Εμφάνιση αποτελεσμάτων
    if not Found:
        print(f"The Keyword '{SearchWords}' couldn't be found in any of our articles.")
    else:
        print(f"The Keyword '{SearchWords}' was found in the Following article(s): ")
        c = 0
        for i in Found:
            c = c + 1
            print(f"{c}. Title: '{Data[i]['Title']}' | URL: [{Data[i]['Url']}] ")
    return Found


#### Συνάρτηση για αναζήτηση λέξεων στο ευρετήριο

In [49]:
def SearchKeyword(SearchWord):
    with open('Data3.json', 'r') as File:
        InvertedIndexes = json.load(File)
    
    for w, i in InvertedIndexes.items():
        if w == SearchWord:
            return i
    return []

#### Υλοποίηση αλγορίθμου Vector Space Model

In [52]:
def VSM(Data, SearchWords):

    # Δημιουργία διανισμάτων TF-IDF για τα κείμενα και το ερώτημα
    TFArray, Vectorizer = CalculateTFIDF(Data)
    SearchWords = Vectorizer.transform([SearchWords])

    # Σύγκριση διανυσμάτων με συνάρτηση συνημιτόνου
    VSMScore = cosine_similarity(SearchWords, TFArray).flatten()
    
    VSMScoreRanked = VSMScore.argsort()
    print(VSMScoreRanked)

    # Ταξινόνιμιση αποτελεσμάτων
    Results = [(i, VSMScore[i]) for i in VSMScoreRanked]
    Results = sorted(Results, key=lambda R:R[1], reverse = True)

    # Εμάνιση αποτελεσμάτων
    c = 0
    for i, VSMScore in Results:
        c = c + 1
        print(f"{c}. Title: '{Data[i]['Title']}' | Score: {VSMScore:.4f}")
        print(f"   URL: [{Data[i]['Url']}]")
    
    return Results

#### Συνάρτηση δημιουργίας πίνακα TF-IDF για την υλοποίηση του VSM

In [55]:
def CalculateTFIDF(Data):

    # Αποθήκευση κειμένων σε πίνακα
    Contents = [Segment['Content'] for Segment in Data]
    Contents = [C.lower() for C in Contents]

    # Μετατρροπή κειμένων σε διανύσματα
    Vectorizer = TfidfVectorizer()
    TFArray = Vectorizer.fit_transform(Contents)
    
    return TFArray, Vectorizer

### Υλοποίηση αλγορίθμου Probabilistic Retrieval

In [61]:
def ProbabilisticRetrieval(Data, SearchWords):

    #αποθήκευσαη λέξεων των κειμένων και μετατροπή σε tokens 
    Contents = [Segment['Content'] for Segment in Data]
    Contents = [C.lower().split(" ") for C in Contents]

    BM25 = BM25Okapi(Contents)

    # μετατροπή ερωτήματος σε tokens
    SearchWords = SearchWords.split(" ")

    # Υπολογισμός βαθμολογίας BM25
    BMScore = BM25.get_scores(SearchWords)

    # Ταξινόμιση αποτελεσμάτων
    Results = [(i, BMScore[i]) for i in range(len(BMScore))]
    Results = sorted(Results, key=lambda R:R[1], reverse = True)

    # Εμφάνιση αποτελεσμάτων
    c = 0
    for i, Score in Results:
        c = c + 1
        print(f"{c}. Title: '{Data[i]['Title']}' | BMScore: {Score}")
        print(f"   URL: [{Data[i]['Url']}]")
    
    return Results

#### Συνάρτηση για ανάλυση του ερωτήματος

In [ ]:
def CheckQuery(SearchWords):
    return re.findall(r'\w+|AND|OR|NOT', SearchWords.lower())

### Υλοποίηση διεπαφής και επεξεργασια του ερωτήματος

In [69]:

def main():
    with open('Data.json', 'r') as file:
        Data = json.load(file)

    print("...Wikipedia Search Engine...")
    print("=============================")
    print("1. Boolean retrieval")
    print("2. Vector Space Model (VSM)")
    print("3. Probabilistic Retrieval")
    Algorithm = input("Choose algorithm to sort the outcomes: ")
    print("=========================================")
    SearchWords = input("Insert keyword(s): ")
    print("================================")

    # Επεξεργασία ερωτήματος 
    SearchWords = SearchWords.lower()
    Query = CheckQuery(SearchWords)

    if Algorithm == "1":
        BooleanRetrieval(Query, len(Data), SearchWords, Data)
    elif Algorithm == "2":
        VSM(Data, SearchWords)
    elif Algorithm == "3":
        ProbabilisticRetrieval(Data, SearchWords)
    else:
        print("No such algorithm")
        main()
        return 0
        
    return 0

if __name__ == "__main__":
    main()

...Wikipedia Search Engine...
1. Boolean retrieval
2. Vector Space Model (VSM)
3. Probabilistic Retrieval


Choose algorithm to sort the outcomes:  1


Insert keyword(s):  real or stylized


The Keyword 'real or stylized' was found in the Following article(s): 
1. Title: 'Radclyffe' | URL: [https://en.wikipedia.org/wiki/Radclyffe] 
2. Title: 'None' | URL: [https://en.wikipedia.org/wiki/Exception_(TV_series)] 


# Μέρος πέμπτο: Αξιολόγιση μηχανής αναζήτησης

In [ ]:
import json
import numpy as np

from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score

import import_ipynb
from Final import VSM, ProbabilisticRetrieval, BooleanRetrieval, CheckQuery

#### Υλοποίηση αξιολόγισης 

In [16]:
def Assessment(Data, Queries, Algorithm):
    # Αρχικοποίηση πινάκων για μετρικές αξιολόγισης 
    PrecisionArray = []
    RecallArray = []
    F1Array = []
    ApArray = []
    
    for Query in Queries:
        Q = Query["0"]
        Results = []

        # Εκτέλεση αντίστοιχης συνάρτησης για κάθε αλγόριθμο
        if Algorithm == "1":
            print(f"Boolean Retrieval results for '{Q}'")
            print("=========================")
            Results = BooleanRetrieval(CheckQuery(Q), len(Data), Q, Data)
            print("=========================")
            print(" ")
        elif Algorithm == "2":
            print(f"Vector Space Model results for '{Q}'")
            print("==========================")
            Results = VSM(Data, Q)
            print("==========================")
            print(" ")
        elif Algorithm == "3":
            print(f"Probabilistic Retrieval results for '{Q}'")
            print("===============================")
            Results = ProbabilisticRetrieval(Data, Q)
            print("===============================")
            print(" ")

        if not Results:
            Results=0
            
        if isinstance(Results, int):
            Results = [Results]
        elif isinstance(Results, list) and isinstance(Results[0], tuple):
            Results = [R[0] for R in Results]

        #Μετατροπή αποτελεσμάτων σε διαδικό διάνυσμα
        Res = [1 if i in Results else 0 for i in range(len(Data))]
        Prediction = [1 if i in Query["1"] else 0 for i in range(len(Data))]

         #Υπολογισμός μετρικών αξιολόγισης
        Precision = precision_score(Res, Prediction)
        Recall = recall_score(Res, Prediction)
        F1 = f1_score(Res, Prediction)
        Ap = average_precision_score(Res, Prediction)

        #Αποθήκευση μετρικών
        PrecisionArray.append(Precision)
        RecallArray.append(Recall)
        F1Array.append(F1)
        ApArray.append(Ap)
    
    print("Assessment results")
    print(f"Precision: {np.mean(PrecisionArray):.4f}")
    print(f"Recall: {np.mean(RecallArray):.4f}")
    print(f"F1-Score: {np.mean(F1Array):.4f}")
    print(f"Mean Average Precision (MAP): {np.mean(ApArray):.4f}")
    return 0



In [18]:
def main():
    with open('Data.json', 'r') as file:
        Data = json.load(file)

    #Δημιουργία ενός ερωτήματος για δοκιμή
    Queries = [
        {"0": "real member", "1": [0,1,2]}
    ]

    #Εκτέλεση της συνάρτησης Assesment για κάθε αλγόριθμο κατάταξης
    Assessment(Data, Queries, "1")
    print("\n")
    Assessment(Data, Queries, "2")
    print("\n")
    Assessment(Data, Queries, "3")

    return 0

if __name__ == "__main__":
    main()

Boolean Retrieval results for 'real member'
The Keyword 'real member' was found in the Following article(s): 
1. Title: 'Radclyffe' | URL: [https://en.wikipedia.org/wiki/Radclyffe] 
2. Title: 'Paula Penacca' | URL: [https://en.wikipedia.org/wiki/Paula_Penacca] 
3. Title: 'None' | URL: [https://en.wikipedia.org/wiki/Exception_(TV_series)] 
 
Assessment results
Precision: 0.3333
Recall: 0.3333
F1-Score: 0.3333
Mean Average Precision (MAP): 0.3111


Vector Space Model results for 'real member'
[0 2 3 4 5 7 8 9 6 1]
1. Title: 'Radclyffe' | Score: 0.0509
   URL: [https://en.wikipedia.org/wiki/Radclyffe]
2. Title: 'Paula Penacca' | Score: 0.0404
   URL: [https://en.wikipedia.org/wiki/Paula_Penacca]
3. Title: 'None' | Score: 0.0190
   URL: [https://en.wikipedia.org/wiki/Exception_(TV_series)]
4. Title: 'None' | Score: 0.0000
   URL: [https://en.wikipedia.org/wiki/Cover_Up_(Ministry_album)]
5. Title: 'Special Box' | Score: 0.0000
   URL: [https://en.wikipedia.org/wiki/Special_Box]
6. Title: 'S